In [ ]:
#Feature generation whose pvalue<0.5 and Correlatoin>0.1

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from scipy.stats import pearsonr

# Load data
df = pd.read_csv('features.csv')

# Apply log transformation to the label
df['label'] = np.log1p(df['label'])  # Use log1p to safely handle zeros

# Separate features and label
features = [col for col in df.columns if col != 'label']
X = df[features]
y = df['label']

# Standardize the feature columns
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Put scaled features back into a DataFrame
X_scaled_df = pd.DataFrame(X_scaled, columns=features)

# Pearson correlation analysis between each feature and the (transformed) label
correlations = []
p_values = []

for feature in features:
    corr, p = pearsonr(X_scaled_df[feature], y)
    correlations.append(corr)
    p_values.append(p)

# Create a DataFrame with correlation results
result_df = pd.DataFrame({
    'Feature': features,
    'Correlation': correlations,
    'P-Value': p_values
})

# Select features based on correlation strength and statistical significance
selected_features = result_df[(result_df['P-Value'] < 0.5) & (result_df['Correlation'].abs() > 0.3)]

# Display selected features
print("Selected features (|correlation| > 0.2 and p < 0.5):")
print(selected_features)

# Get list of selected feature names
selected_columns = selected_features['Feature'].tolist()

# Final filtered DataFrame (scaled features + log-transformed label)
df_filtered = X_scaled_df[selected_columns]
df_filtered['label'] = y  # Add back the log-transformed label

# Save to CSV
df_filtered.to_csv('selected_features.csv', index=False)
print("Saved selected and preprocessed features to 'selected_features.csv'")

✅ Selected features (|correlation| > 0.2 and p < 0.5):
    Feature  Correlation       P-Value
425   BTC_T     0.328353  1.083972e-27
426   BTC_H     0.327475  1.521895e-27
427   BTC_S     0.328159  1.168573e-27
428   BTC_D     0.329940  5.856248e-28
434   RRI_G     0.327278  1.641444e-27
435   RRI_H     0.315289  1.502759e-25
443   RRI_R     0.336949  3.694213e-29
447   RRI_W     0.449724  3.640042e-53
448   RRI_Y     0.324052  5.647111e-27
451   DDR_D     0.377264  1.097753e-36
452   DDR_E     0.314586  1.946083e-25
453   DDR_F     0.335135  7.605188e-29
454   DDR_G     0.377101  1.183664e-36
455   DDR_H     0.368225  6.660189e-35
456   DDR_I     0.339687  1.230864e-29
457   DDR_K     0.341753  5.333395e-30
459   DDR_M     0.309145  1.406306e-24
460   DDR_N     0.313342  3.070325e-25
461   DDR_P     0.316330  1.023534e-25
462   DDR_Q     0.355404  1.805163e-32
463   DDR_R     0.344936  1.451175e-30
464   DDR_S     0.351910  7.950118e-32
465   DDR_T     0.367960  7.497770e-35
467   DDR

/tmp/ipython-input-2566134859.py:54: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered['label'] = y  # Add back the log-transformed label


In [ ]:

import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np


X = df_filtered.drop('label', axis=1)
y = df_filtered['label']

# Try with 5 bins (fewer bins to avoid empty or single-sample bins)
num_bins = 5
y_binned = pd.cut(y, bins=num_bins, labels=False)

# Check bin counts (optional debug)
print(pd.Series(y_binned).value_counts())

# Split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y_binned
)

label
0    376
2    303
3    213
1    126
4     27
Name: count, dtype: int64


In [ ]:
X_train.to_csv('X_train.csv', index=False)
y_train.to_csv('y_train.csv', index=False)
X_test.to_csv('X_test.csv', index=False)
y_test.to_csv('y_test.csv', index=False)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import lightgbm as lgb
from scipy.stats import pearsonr

# Load preprocessed dataset
df = pd.read_csv('selected_features.csv')

# Features and label
X = df.drop('label', axis=1)
y_log = df['label']  # already log-transformed

# Standardize label
y_scaler = StandardScaler()
y_scaled = y_scaler.fit_transform(y_log.values.reshape(-1, 1)).flatten()

# Stratified binning for split
num_bins = 5
y_binned = pd.cut(y_scaled, bins=num_bins, labels=False)

# Split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y_scaled, test_size=0.2, random_state=42, stratify=y_binned
)

# Define models
models = {
    "SVR": SVR(),
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(),
    "Random Forest": RandomForestRegressor(),
    "Gradient Boosting": GradientBoostingRegressor(),
    "AdaBoost": AdaBoostRegressor(),
    "K-Nearest Neighbors": KNeighborsRegressor(),
    "XGBoost": xgb.XGBRegressor(),
    "LightGBM": lgb.LGBMRegressor()
}

# Parameter grid
param_grid = {
    'SVR': {'C': [0.1, 1, 10], 'epsilon': [0.01, 0.1], 'gamma': ['scale', 'auto']},
    'Random Forest': {'n_estimators': [50, 100, 200], 'max_depth': [None, 10, 20]},
    'XGBoost': {'n_estimators': [50, 100, 200], 'learning_rate': [0.01, 0.1, 0.2]},
    'LightGBM': {'n_estimators': [50, 100, 200], 'learning_rate': [0.01, 0.1, 0.2]}
}

# Helper function to safely apply inverse_transform and expm1
def safe_inverse_expm1(y_scaled, scaler, clip_max=100):
    y_scaled = np.nan_to_num(y_scaled, nan=0.0, posinf=1e6, neginf=-1e6)
    y_unscaled = scaler.inverse_transform(y_scaled.reshape(-1, 1)).flatten()
    y_unscaled = np.clip(y_unscaled, a_min=None, a_max=clip_max)
    return np.expm1(y_unscaled)

# Evaluation function
def evaluate_models(models, param_grid, X_train, y_train, X_test, y_test):
    results = []
    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    for model_name, model in models.items():
        print(f"\nTraining {model_name}...")

        grid_search = GridSearchCV(
            estimator=model,
            param_grid=param_grid.get(model_name, {}),
            cv=kf,
            scoring='neg_mean_squared_error',
            return_train_score=True
        )
        grid_search.fit(X_train, y_train)

        best_model = grid_search.best_estimator_
        print(f"Best parameters for {model_name}: {grid_search.best_params_}")

        train_metrics = {'mse': [], 'mae': [], 'rmse': [], 'pcc': [], 'r2': []}
        val_metrics = {'mse': [], 'mae': [], 'rmse': [], 'pcc': [], 'r2': []}

        for train_idx, val_idx in kf.split(X_train):
            X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
            y_tr, y_val = y_train[train_idx], y_train[val_idx]

            best_model.fit(X_tr, y_tr)

            y_tr_pred_scaled = best_model.predict(X_tr)
            y_val_pred_scaled = best_model.predict(X_val)

            y_tr_pred = safe_inverse_expm1(y_tr_pred_scaled, y_scaler)
            y_val_pred = safe_inverse_expm1(y_val_pred_scaled, y_scaler)
            y_tr_orig = safe_inverse_expm1(y_tr, y_scaler)
            y_val_orig = safe_inverse_expm1(y_val, y_scaler)

            train_metrics['mse'].append(mean_squared_error(y_tr_orig, y_tr_pred))
            train_metrics['mae'].append(mean_absolute_error(y_tr_orig, y_tr_pred))
            train_metrics['rmse'].append(np.sqrt(mean_squared_error(y_tr_orig, y_tr_pred)))
            train_metrics['r2'].append(r2_score(y_tr_orig, y_tr_pred))
            train_metrics['pcc'].append(pearsonr(y_tr_orig, y_tr_pred)[0])

            val_metrics['mse'].append(mean_squared_error(y_val_orig, y_val_pred))
            val_metrics['mae'].append(mean_absolute_error(y_val_orig, y_val_pred))
            val_metrics['rmse'].append(np.sqrt(mean_squared_error(y_val_orig, y_val_pred)))
            val_metrics['r2'].append(r2_score(y_val_orig, y_val_pred))
            val_metrics['pcc'].append(pearsonr(y_val_orig, y_val_pred)[0])

        best_model.fit(X_train, y_train)
        y_test_pred_scaled = best_model.predict(X_test)

        y_test_pred = safe_inverse_expm1(y_test_pred_scaled, y_scaler)
        y_test_orig = safe_inverse_expm1(y_test, y_scaler)

        test_mse = mean_squared_error(y_test_orig, y_test_pred)
        test_mae = mean_absolute_error(y_test_orig, y_test_pred)
        test_rmse = np.sqrt(test_mse)
        test_r2 = r2_score(y_test_orig, y_test_pred)
        test_pcc = pearsonr(y_test_orig, y_test_pred)[0]

        result = {
            'Regressor': model_name,
            'Training MSE': np.mean(train_metrics['mse']),
            'Training MAE': np.mean(train_metrics['mae']),
            'Training RMSE': np.mean(train_metrics['rmse']),
            'Training PCC': np.mean(train_metrics['pcc']),
            'Training R2': np.mean(train_metrics['r2']),
            'Validation MSE': test_mse,
            'Validation MAE': test_mae,
            'Validation RMSE': test_rmse,
            'Validation PCC': test_pcc,
            'Validation R2': test_r2
        }
        results.append(result)

    return pd.DataFrame(results)

# Run evaluation
performance_results = evaluate_models(models, param_grid, X_train, y_train, X_test, y_test)

# Format and export results
formatted_results = performance_results[[
    'Regressor',
    'Training MSE', 'Training MAE', 'Training RMSE', 'Training PCC', 'Training R2',
    'Validation MSE', 'Validation MAE', 'Validation RMSE', 'Validation PCC', 'Validation R2'
]]

# Round numeric values
formatted_results[formatted_results.select_dtypes(include=[np.number]).columns] = \
    formatted_results.select_dtypes(include=[np.number]).round(2)

# Save to CSV
formatted_results.to_csv('model_performance_80_20.csv', index=False)

# Display results
print("\nModel Performance (5-Fold CV on 80% Training, Tested on 20%):\n")
print(formatted_results.to_string(index=False))



Training SVR...
Best parameters for SVR: {'C': 10, 'epsilon': 0.1, 'gamma': 'scale'}

Training Linear Regression...
Best parameters for Linear Regression: {}

Training Ridge...
Best parameters for Ridge: {}

Training Lasso...
Best parameters for Lasso: {}

Training Random Forest...


/tmp/ipython-input-1198175456.py:104: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  train_metrics['pcc'].append(pearsonr(y_tr_orig, y_tr_pred)[0])
/tmp/ipython-input-1198175456.py:110: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  val_metrics['pcc'].append(pearsonr(y_val_orig, y_val_pred)[0])
/tmp/ipython-input-1198175456.py:104: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  train_metrics['pcc'].append(pearsonr(y_tr_orig, y_tr_pred)[0])
/tmp/ipython-input-1198175456.py:110: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  val_metrics['pcc'].append(pearsonr(y_val_orig, y_val_pred)[0])
/tmp/ipython-input-1198175456.py:104: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  train_metrics['pcc'].append(pearsonr(y_tr_orig, y_tr_pred)[0])
/tmp/ipython-in

Streaming output truncated to the last 5000 lines.
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posit

In [ ]:
# Deep Learning

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim

# CONFIG

DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
EPOCHS     = 100
LR         = 1e-3
PATIENCE   = 15          # early-stopping patience
SEED       = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
print(f"Using device: {DEVICE}")

# 1. LOAD & PREPARE DATA  (same as your original script)

df = pd.read_csv("selected_features.csv")

X_raw = df.drop("label", axis=1).values.astype(np.float32)
y_log = df["label"].values.astype(np.float32)       # already log-transformed

# Standardise features
feat_scaler = StandardScaler()
X_raw = feat_scaler.fit_transform(X_raw)

# Standardise label
y_scaler = StandardScaler()
y_scaled  = y_scaler.fit_transform(y_log.reshape(-1, 1)).flatten().astype(np.float32)

# Stratified split
num_bins = 5
y_binned = pd.cut(y_scaled, bins=num_bins, labels=False)

X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y_scaled, test_size=0.2, random_state=SEED, stratify=y_binned
)

INPUT_DIM = X_train.shape[1]

# HELPER: inverse transform  (mirrors safe_inverse_expm1)

def safe_inverse_expm1(y_s: np.ndarray, clip_max: float = 100.0) -> np.ndarray:
    y_s = np.nan_to_num(y_s, nan=0.0, posinf=1e6, neginf=-1e6)
    y_u = y_scaler.inverse_transform(y_s.reshape(-1, 1)).flatten()
    y_u = np.clip(y_u, a_min=None, a_max=clip_max)
    return np.expm1(y_u)


def compute_metrics(y_true_orig, y_pred_orig):
    mse  = mean_squared_error(y_true_orig, y_pred_orig)
    mae  = mean_absolute_error(y_true_orig, y_pred_orig)
    rmse = np.sqrt(mse)
    r2   = r2_score(y_true_orig, y_pred_orig)
    pcc  = pearsonr(y_true_orig, y_pred_orig)[0]
    return mse, mae, rmse, pcc, r2

# PYTORCH MODEL ARCHITECTURES

class MLP(nn.Module):
    """Plain feed-forward MLP with BatchNorm + Dropout."""
    def __init__(self, in_dim, hidden=[256, 128, 64], dropout=0.3):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


class ResidualBlock(nn.Module):
    """Two-layer residual block with optional projection."""
    def __init__(self, dim, dropout=0.3):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim), nn.BatchNorm1d(dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(dim, dim), nn.BatchNorm1d(dim)
        )
        self.act = nn.ReLU()

    def forward(self, x):
        return self.act(x + self.block(x))


class ResidualMLP(nn.Module):
    """Deep MLP with skip connections — better gradient flow."""
    def __init__(self, in_dim, hidden=256, n_blocks=4, dropout=0.3):
        super().__init__()
        self.stem    = nn.Sequential(nn.Linear(in_dim, hidden), nn.BatchNorm1d(hidden), nn.ReLU())
        self.blocks  = nn.Sequential(*[ResidualBlock(hidden, dropout) for _ in range(n_blocks)])
        self.head    = nn.Linear(hidden, 1)

    def forward(self, x):
        return self.head(self.blocks(self.stem(x))).squeeze(-1)


class CNN1DRegressor(nn.Module):
    """
    Treats each feature as a 1-D 'channel' and applies temporal convolutions.
    Useful when features carry sequential / grouped structure.
    """
    def __init__(self, in_dim, channels=[64, 128, 64], kernel=3, dropout=0.3):
        super().__init__()
        convs = []
        in_c  = 1
        for out_c in channels:
            convs += [
                nn.Conv1d(in_c, out_c, kernel_size=kernel, padding=kernel // 2),
                nn.BatchNorm1d(out_c), nn.ReLU(), nn.Dropout(dropout)
            ]
            in_c = out_c
        self.convs   = nn.Sequential(*convs)
        self.pool    = nn.AdaptiveAvgPool1d(1)
        self.head    = nn.Linear(channels[-1], 1)

    def forward(self, x):                       # x: (B, F)
        x = x.unsqueeze(1)                       # (B, 1, F)
        x = self.pool(self.convs(x)).squeeze(-1) # (B, C)
        return self.head(x).squeeze(-1)


class TransformerRegressor(nn.Module):
    """
    Wraps each feature as a token and applies multi-head self-attention.
    Captures feature interactions without assuming spatial locality.
    """
    def __init__(self, in_dim, d_model=64, nhead=4, n_layers=2, dropout=0.1):
        super().__init__()
        self.embed    = nn.Linear(1, d_model)
        enc_layer     = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward=128,
                                                    dropout=dropout, batch_first=True)
        self.encoder  = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.head     = nn.Linear(d_model, 1)

    def forward(self, x):                        # x: (B, F)
        x = self.embed(x.unsqueeze(-1))          # (B, F, d_model)
        x = self.encoder(x)                      # (B, F, d_model)
        x = x.mean(dim=1)                        # (B, d_model)
        return self.head(x).squeeze(-1)

# TRAINING UTILITIES

def make_loader(X, y, shuffle=True):
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32),
                       torch.tensor(y, dtype=torch.float32))
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle)


def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(yb)
    return total / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total, preds = 0.0, []
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        out = model(xb)
        total += criterion(out, yb).item() * len(yb)
        preds.append(out.cpu().numpy())
    return total / len(loader.dataset), np.concatenate(preds)


def train_model(model, X_tr, y_tr, X_val, y_val):
    """Full training loop with early stopping. Returns best predictions on val."""
    model.to(DEVICE)
    criterion = nn.MSELoss()
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

    tr_loader  = make_loader(X_tr, y_tr, shuffle=True)
    val_loader = make_loader(X_val, y_val, shuffle=False)

    best_val, best_preds, no_improve = np.inf, None, 0
    best_state = None

    for epoch in range(1, EPOCHS + 1):
        train_one_epoch(model, tr_loader, optimizer, criterion)
        val_loss, val_preds = evaluate(model, val_loader, criterion)
        scheduler.step(val_loss)

        if val_loss < best_val:
            best_val   = val_loss
            best_preds = val_preds.copy()
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                break

    model.load_state_dict(best_state)
    return model, best_preds

# EVALUATION LOOP  (5-Fold CV, same as original)

def get_model(name):
    constructors = {
        "MLP":                 lambda: MLP(INPUT_DIM),
        "Residual MLP":        lambda: ResidualMLP(INPUT_DIM),
        "1D-CNN":              lambda: CNN1DRegressor(INPUT_DIM),
        "Transformer Encoder": lambda: TransformerRegressor(INPUT_DIM),
    }
    return constructors[name]()


dl_models = ["MLP", "Residual MLP", "1D-CNN", "Transformer Encoder"]


def evaluate_dl_models(model_names, X_train, y_train, X_test, y_test):
    results = []
    kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

    for name in model_names:
        print(f"\n{'='*50}\nTraining {name} ...")
        tr_m = {k: [] for k in ["mse", "mae", "rmse", "pcc", "r2"]}
        va_m = {k: [] for k in ["mse", "mae", "rmse", "pcc", "r2"]}

        for fold, (ti, vi) in enumerate(kf.split(X_train), 1):
            X_tr, X_val = X_train[ti], X_train[vi]
            y_tr, y_val = y_train[ti], y_train[vi]

            model = get_model(name)
            model, val_preds = train_model(model, X_tr, y_tr, X_val, y_val)

            # Training predictions
            _, tr_preds = evaluate(model, make_loader(X_tr, y_tr, False), nn.MSELoss())

            # Inverse transform back to original scale
            y_tr_orig  = safe_inverse_expm1(y_tr)
            y_val_orig = safe_inverse_expm1(y_val)
            y_tr_pred  = safe_inverse_expm1(tr_preds)
            y_val_pred = safe_inverse_expm1(val_preds)

            for k, v in zip(["mse","mae","rmse","r2","pcc"],
                            compute_metrics(y_tr_orig, y_tr_pred)):
                tr_m[k].append(v)
            for k, v in zip(["mse","mae","rmse","r2","pcc"],
                            compute_metrics(y_val_orig, y_val_pred)):
                va_m[k].append(v)

            print(f"  Fold {fold} — Val RMSE: {va_m['rmse'][-1]:.4f}  R²: {va_m['r2'][-1]:.4f}")

        # --- Final model: retrain on full training set, evaluate on held-out test ---
        model = get_model(name)
        model, _ = train_model(model, X_train, y_train, X_test, y_test)  # uses test for early-stop only

        _, test_preds = evaluate(model, make_loader(X_test, y_test, False), nn.MSELoss())
        y_test_orig = safe_inverse_expm1(y_test)
        y_test_pred = safe_inverse_expm1(test_preds)
        t_mse, t_mae, t_rmse, t_pcc, t_r2 = compute_metrics(y_test_orig, y_test_pred)

        results.append({
            "Regressor":       name,
            "Training MSE":    round(np.mean(tr_m["mse"]),  4),
            "Training MAE":    round(np.mean(tr_m["mae"]),  4),
            "Training RMSE":   round(np.mean(tr_m["rmse"]), 4),
            "Training PCC":    round(np.mean(tr_m["pcc"]),  4),
            "Training R2":     round(np.mean(tr_m["r2"]),   4),
            "Validation MSE":  round(t_mse,  4),
            "Validation MAE":  round(t_mae,  4),
            "Validation RMSE": round(t_rmse, 4),
            "Validation PCC":  round(t_pcc,  4),
            "Validation R2":   round(t_r2,   4),
        })
        print(f"  Test  — RMSE: {t_rmse:.4f}  R²: {t_r2:.4f}  PCC: {t_pcc:.4f}")

    return pd.DataFrame(results)

# RUN

if __name__ == "__main__":
    dl_results = evaluate_dl_models(dl_models, X_train, y_train, X_test, y_test)
    dl_results.to_csv("dl_model_performance.csv", index=False)
    print("\n\nDeep Learning Model Performance:\n")
    print(dl_results.to_string(index=False))

In [ ]:
# ProtBert Model 
import argparse
import re
import time
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from transformers import BertTokenizer, BertModel

# CONFIG

PROTBERT_MODEL = "Rostlab/prot_bert_bfd"   # or "Rostlab/prot_bert"
MAX_LEN        = 512                        # ProtBERT max tokens
BATCH_SIZE_EMB = 8                          # for embedding extraction
BATCH_SIZE_FT  = 4                          # for fine-tuning (large model)
EPOCHS_HEAD    = 80                         # MLP head epochs (Mode A)
EPOCHS_FT      = 20                         # fine-tune epochs (Mode B)
LR_BERT        = 2e-5                       # BERT backbone lr
LR_HEAD        = 1e-3                       # regression head lr
PATIENCE       = 10
SEED           = 42
DEVICE         = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(SEED)
np.random.seed(SEED)
print(f"Using device : {DEVICE}")
print(f"ProtBERT model: {PROTBERT_MODEL}")

# 1. LOAD DATA

df = pd.read_csv("selected_features.csv")

# Sequence column 
# Adjust column name if yours is different (e.g., 'protein_seq', 'aa_seq')
if "sequence" not in df.columns:
    raise ValueError(
        "CSV must contain a 'sequence' column with raw amino-acid sequences.\n"
        "Rename your sequence column to 'sequence' or update this script."
    )

sequences = df["sequence"].tolist()
y_log     = df["label"].values.astype(np.float32)

y_scaler = StandardScaler()
y_scaled = y_scaler.fit_transform(y_log.reshape(-1, 1)).flatten().astype(np.float32)

num_bins = 5
y_binned = pd.cut(y_scaled, bins=num_bins, labels=False)

idx = np.arange(len(sequences))
tr_idx, te_idx = train_test_split(idx, test_size=0.2, random_state=SEED, stratify=y_binned)

seq_train  = [sequences[i] for i in tr_idx]
seq_test   = [sequences[i] for i in te_idx]
y_train    = y_scaled[tr_idx]
y_test     = y_scaled[te_idx]

# HELPERS


def preprocess_seq(seq: str) -> str:
    """ProtBERT requires space-separated amino acids and unknown AA → X."""
    seq = re.sub(r"[UZOB]", "X", seq.upper())
    return " ".join(list(seq))


def safe_inverse_expm1(y_s: np.ndarray, clip_max: float = 100.0) -> np.ndarray:
    y_s = np.nan_to_num(y_s, nan=0.0, posinf=1e6, neginf=-1e6)
    y_u = y_scaler.inverse_transform(y_s.reshape(-1, 1)).flatten()
    return np.expm1(np.clip(y_u, None, clip_max))


def compute_metrics(true, pred):
    mse  = mean_squared_error(true, pred)
    mae  = mean_absolute_error(true, pred)
    rmse = np.sqrt(mse)
    r2   = r2_score(true, pred)
    pcc  = pearsonr(true, pred)[0]
    return mse, mae, rmse, pcc, r2

# LOAD TOKENIZER & MODEL

print("\nLoading ProtBERT tokenizer and model …")
tokenizer = BertTokenizer.from_pretrained(PROTBERT_MODEL, do_lower_case=False)
bert_model = BertModel.from_pretrained(PROTBERT_MODEL)
bert_model.to(DEVICE)
print("ProtBERT loaded.")

# FEATURE EXTRACTION  (embed once, then train MLP head)

@torch.no_grad()
def extract_embeddings(seqs, batch_size=BATCH_SIZE_EMB):
    """Return mean-pooled [CLS]-excluded embeddings; shape (N, 1024)."""
    bert_model.eval()
    all_embs = []
    for i in range(0, len(seqs), batch_size):
        batch = [preprocess_seq(s) for s in seqs[i: i + batch_size]]
        enc   = tokenizer(batch, return_tensors="pt", padding=True,
                          truncation=True, max_length=MAX_LEN).to(DEVICE)
        out   = bert_model(**enc)
        # Mean-pool over sequence tokens (exclude [CLS]=0 and [SEP]=last)
        mask  = enc["attention_mask"].unsqueeze(-1).float()
        emb   = (out.last_hidden_state * mask).sum(1) / mask.sum(1)
        all_embs.append(emb.cpu().numpy())
        if (i // batch_size) % 10 == 0:
            print(f"  Embedded {min(i+batch_size, len(seqs))}/{len(seqs)}")
    return np.vstack(all_embs).astype(np.float32)


class MLPHead(nn.Module):
    def __init__(self, in_dim=1024, hidden=[512, 256, 128], dropout=0.3):
        super().__init__()
        layers, prev = [], in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.LayerNorm(h), nn.GELU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


def train_mlp_head(X_tr, y_tr, X_val, y_val, in_dim=1024):
    model     = MLPHead(in_dim).to(DEVICE)
    criterion = nn.MSELoss()
    opt       = AdamW(model.parameters(), lr=LR_HEAD, weight_decay=1e-4)
    sched     = CosineAnnealingLR(opt, T_max=EPOCHS_HEAD)

    tr_ds  = torch.utils.data.TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr))
    val_ds = torch.utils.data.TensorDataset(torch.tensor(X_val), torch.tensor(y_val))
    tr_dl  = DataLoader(tr_ds, batch_size=32, shuffle=True)
    val_dl = DataLoader(val_ds, batch_size=32)

    best_val, best_state, no_imp = np.inf, None, 0
    for epoch in range(1, EPOCHS_HEAD + 1):
        model.train()
        for xb, yb in tr_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            criterion(model(xb), yb).backward()
            opt.step()
        sched.step()

        model.eval()
        val_preds = []
        val_loss  = 0
        with torch.no_grad():
            for xb, yb in val_dl:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                out = model(xb)
                val_loss += criterion(out, yb).item() * len(yb)
                val_preds.append(out.cpu().numpy())
        val_loss /= len(val_dl.dataset)

        if val_loss < best_val:
            best_val   = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            best_preds = np.concatenate(val_preds)
            no_imp     = 0
        else:
            no_imp += 1
            if no_imp >= PATIENCE:
                break

    model.load_state_dict(best_state)
    return model, best_preds


def run_mode_A():
    print("\n" + "="*60)
    print("MODE A — ProtBERT feature extraction + MLP head (5-Fold CV)")
    print("="*60)

    print("\nExtracting embeddings for training set …")
    X_tr_emb = extract_embeddings(seq_train)
    print("Extracting embeddings for test set …")
    X_te_emb = extract_embeddings(seq_test)

    # L2-normalise embeddings
    norm = np.linalg.norm(X_tr_emb, axis=1, keepdims=True) + 1e-8
    X_tr_emb /= norm
    X_te_emb /= (np.linalg.norm(X_te_emb, axis=1, keepdims=True) + 1e-8)

    IN_DIM = X_tr_emb.shape[1]
    kf     = KFold(n_splits=5, shuffle=True, random_state=SEED)
    tr_m   = {k: [] for k in ["mse","mae","rmse","pcc","r2"]}
    va_m   = {k: [] for k in ["mse","mae","rmse","pcc","r2"]}

    for fold, (ti, vi) in enumerate(kf.split(X_tr_emb), 1):
        model, val_preds = train_mlp_head(
            X_tr_emb[ti], y_train[ti],
            X_tr_emb[vi], y_train[vi],
            in_dim=IN_DIM
        )
        # training preds
        model.eval()
        with torch.no_grad():
            tr_preds = model(torch.tensor(X_tr_emb[ti]).to(DEVICE)).cpu().numpy()

        for m, v in zip(["mse","mae","rmse","pcc","r2"],
                        compute_metrics(safe_inverse_expm1(y_train[ti]),
                                        safe_inverse_expm1(tr_preds))):
            tr_m[m].append(v)
        for m, v in zip(["mse","mae","rmse","pcc","r2"],
                        compute_metrics(safe_inverse_expm1(y_train[vi]),
                                        safe_inverse_expm1(val_preds))):
            va_m[m].append(v)

        print(f"  Fold {fold} — Val RMSE: {va_m['rmse'][-1]:.4f}  R²: {va_m['r2'][-1]:.4f}")

    # Final: train on full train set, test on held-out
    final_model, _ = train_mlp_head(X_tr_emb, y_train, X_te_emb, y_test, in_dim=IN_DIM)
    final_model.eval()
    with torch.no_grad():
        test_preds = final_model(torch.tensor(X_te_emb).to(DEVICE)).cpu().numpy()

    t_mse, t_mae, t_rmse, t_pcc, t_r2 = compute_metrics(
        safe_inverse_expm1(y_test), safe_inverse_expm1(test_preds))

    result = {
        "Regressor":       "ProtBERT + MLP Head",
        "Training MSE":    round(np.mean(tr_m["mse"]),  4),
        "Training MAE":    round(np.mean(tr_m["mae"]),  4),
        "Training RMSE":   round(np.mean(tr_m["rmse"]), 4),
        "Training PCC":    round(np.mean(tr_m["pcc"]),  4),
        "Training R2":     round(np.mean(tr_m["r2"]),   4),
        "Validation MSE":  round(t_mse,  4),
        "Validation MAE":  round(t_mae,  4),
        "Validation RMSE": round(t_rmse, 4),
        "Validation PCC":  round(t_pcc,  4),
        "Validation R2":   round(t_r2,   4),
    }
    print(f"\nTest — RMSE: {t_rmse:.4f}  R²: {t_r2:.4f}  PCC: {t_pcc:.4f}")
    return pd.DataFrame([result])

# END-TO-END FINE-TUNING

class ProteinDataset(Dataset):
    def __init__(self, sequences, labels, tokenizer, max_len=MAX_LEN):
        self.seqs   = [preprocess_seq(s) for s in sequences]
        self.labels = labels
        self.tok    = tokenizer
        self.max_len= max_len

    def __len__(self):
        return len(self.seqs)

    def __getitem__(self, idx):
        enc = self.tok(self.seqs[idx], return_tensors="pt", truncation=True,
                       max_length=self.max_len, padding="max_length")
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label":          torch.tensor(self.labels[idx], dtype=torch.float32)
        }


class ProtBERTRegressor(nn.Module):
    def __init__(self, bert, hidden=256, dropout=0.2):
        super().__init__()
        self.bert = bert
        hidden_size = bert.config.hidden_size        # 1024 for prot_bert_bfd
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1)
        )

    def forward(self, input_ids, attention_mask):
        out  = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        mask = attention_mask.unsqueeze(-1).float()
        pooled = (out.last_hidden_state * mask).sum(1) / mask.sum(1)
        return self.head(pooled).squeeze(-1)


def make_ft_loader(seqs, labels, shuffle=True):
    ds = ProteinDataset(seqs, labels, tokenizer)
    return DataLoader(ds, batch_size=BATCH_SIZE_FT, shuffle=shuffle,
                      num_workers=0, pin_memory=True)


def train_finetune(model, tr_seqs, tr_labels, val_seqs, val_labels):
    criterion = nn.MSELoss()

    # Differential learning rates: lower for BERT, higher for head
    optimizer = AdamW([
        {"params": model.bert.parameters(),  "lr": LR_BERT,  "weight_decay": 0.01},
        {"params": model.head.parameters(),  "lr": LR_HEAD,  "weight_decay": 1e-4},
    ])
    scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS_FT)

    tr_dl  = make_ft_loader(tr_seqs, tr_labels, shuffle=True)
    val_dl = make_ft_loader(val_seqs, val_labels, shuffle=False)

    best_val, best_state, no_imp = np.inf, None, 0
    best_val_preds = None

    for epoch in range(1, EPOCHS_FT + 1):
        model.train()
        t0 = time.time()
        for batch in tr_dl:
            ids  = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            y    = batch["label"].to(DEVICE)
            optimizer.zero_grad()
            criterion(model(ids, mask), y).backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)   # gradient clipping
            optimizer.step()
        scheduler.step()

        # Validation
        model.eval()
        val_loss, val_preds = 0.0, []
        with torch.no_grad():
            for batch in val_dl:
                ids  = batch["input_ids"].to(DEVICE)
                mask = batch["attention_mask"].to(DEVICE)
                y    = batch["label"].to(DEVICE)
                out  = model(ids, mask)
                val_loss += criterion(out, y).item() * len(y)
                val_preds.append(out.cpu().numpy())
        val_loss /= len(val_dl.dataset)
        val_preds = np.concatenate(val_preds)

        print(f"  Epoch {epoch:02d}/{EPOCHS_FT}  val_loss={val_loss:.4f}  "
              f"({time.time()-t0:.0f}s)")

        if val_loss < best_val:
            best_val       = val_loss
            best_val_preds = val_preds.copy()
            best_state     = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_imp         = 0
        else:
            no_imp += 1
            if no_imp >= PATIENCE:
                print("  Early stopping.")
                break

    model.load_state_dict(best_state)
    return model, best_val_preds


def run_mode_B():
    print("\n" + "="*60)
    print("MODE B — ProtBERT end-to-end fine-tuning (5-Fold CV)")
    print("="*60)

    kf   = KFold(n_splits=5, shuffle=True, random_state=SEED)
    tr_m = {k: [] for k in ["mse","mae","rmse","pcc","r2"]}
    va_m = {k: [] for k in ["mse","mae","rmse","pcc","r2"]}

    for fold, (ti, vi) in enumerate(kf.split(seq_train), 1):
        print(f"\n── Fold {fold} ──")
        model = ProtBERTRegressor(BertModel.from_pretrained(PROTBERT_MODEL)).to(DEVICE)

        tr_seqs = [seq_train[i] for i in ti]
        val_seqs= [seq_train[i] for i in vi]

        model, val_preds = train_finetune(model, tr_seqs, y_train[ti],
                                          val_seqs, y_train[vi])

        # Training preds
        model.eval()
        tr_preds = []
        with torch.no_grad():
            for batch in make_ft_loader(tr_seqs, y_train[ti], False):
                ids  = batch["input_ids"].to(DEVICE)
                mask = batch["attention_mask"].to(DEVICE)
                tr_preds.append(model(ids, mask).cpu().numpy())
        tr_preds = np.concatenate(tr_preds)

        for m, v in zip(["mse","mae","rmse","pcc","r2"],
                        compute_metrics(safe_inverse_expm1(y_train[ti]),
                                        safe_inverse_expm1(tr_preds))):
            tr_m[m].append(v)
        for m, v in zip(["mse","mae","rmse","pcc","r2"],
                        compute_metrics(safe_inverse_expm1(y_train[vi]),
                                        safe_inverse_expm1(val_preds))):
            va_m[m].append(v)

        print(f"  Fold {fold} — Val RMSE: {va_m['rmse'][-1]:.4f}  R²: {va_m['r2'][-1]:.4f}")
        del model; torch.cuda.empty_cache()

    # Final model
    print("\nTraining final ProtBERT model on full training set …")
    final_model = ProtBERTRegressor(BertModel.from_pretrained(PROTBERT_MODEL)).to(DEVICE)
    final_model, _ = train_finetune(final_model, seq_train, y_train, seq_test, y_test)

    final_model.eval()
    test_preds = []
    with torch.no_grad():
        for batch in make_ft_loader(seq_test, y_test, False):
            ids  = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            test_preds.append(final_model(ids, mask).cpu().numpy())
    test_preds = np.concatenate(test_preds)

    t_mse, t_mae, t_rmse, t_pcc, t_r2 = compute_metrics(
        safe_inverse_expm1(y_test), safe_inverse_expm1(test_preds))

    result = {
        "Regressor":       "ProtBERT Fine-tuned",
        "Training MSE":    round(np.mean(tr_m["mse"]),  4),
        "Training MAE":    round(np.mean(tr_m["mae"]),  4),
        "Training RMSE":   round(np.mean(tr_m["rmse"]), 4),
        "Training PCC":    round(np.mean(tr_m["pcc"]),  4),
        "Training R2":     round(np.mean(tr_m["r2"]),   4),
        "Validation MSE":  round(t_mse,  4),
        "Validation MAE":  round(t_mae,  4),
        "Validation RMSE": round(t_rmse, 4),
        "Validation PCC":  round(t_pcc,  4),
        "Validation R2":   round(t_r2,   4),
    }
    print(f"\nTest — RMSE: {t_rmse:.4f}  R²: {t_r2:.4f}  PCC: {t_pcc:.4f}")
    return pd.DataFrame([result])

# 7. ENTRY POINT

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--mode", choices=["A", "B", "both"], default="A",
                        help="A=feature extraction, B=fine-tune, both=run both")
    args = parser.parse_args()

    all_results = []
    if args.mode in ("A", "both"):
        all_results.append(run_mode_A())
    if args.mode in ("B", "both"):
        all_results.append(run_mode_B())

    final_df = pd.concat(all_results, ignore_index=True)
    out_path = "protbert_performance.csv"
    final_df.to_csv(out_path, index=False)
    print(f"\nResults saved → {out_path}")
    print(final_df.to_string(index=False))